# Week 3 — Model Improvement

Takes Week 2's baseline random forests (`Featurization_Baseline.ipynb`, kept sealed/unedited as a stable "before" reference) and runs each property through the same sequence of improvements/analyses the project plan calls for. Roadmap for this notebook:

1. **Add TPSA as a 6th descriptor** (all 5 properties) — Week 2's `featurize()` deliberately left it out; TPSA was the single cleanest signal found anywhere in Week 1 EDA.
2. **XGBoost vs. Random Forest** (all 5 properties) — a gradient-boosted alternative to the baseline, compared honestly (not every property picks XGBoost as the winner).
3. **Hyperparameter tuning** (all 5 properties) — `RandomizedSearchCV` with scaffold-grouped CV (`GroupKFold` on Bemis-Murcko scaffolds), scored on each property's own correct TDC leaderboard metric.
4. **Permutation importance** (all 5 properties, tuned XGBoost) — the plan's feature-importance step, using a method that doesn't share Random Forest's default MDI/Gini importance bias (discovered and documented in step 1's investigation).
5. **Failure mode analysis** (all 5 properties, tuned XGBoost) — tests the plan's own suggested hypothesis (does the model do worse on chemically novel test molecules?) directly, then digs into the actual worst predictions regardless of whether that hypothesis holds.

Clearance gets one extra step first: 16%/11% of its rows are pinned at assay floor/ceiling values (`Y=3.0`/`150.0`), not real measurements (flagged in Week 1 EDA). That gets explicitly decided and handled (filtering) before Clearance goes through steps 2-5, in its own section partway through this notebook.

Each property section re-runs the same scaffold-split → featurize → train → evaluate pattern and compares directly against Week 2's frozen baseline numbers. Full narrative writeups (the "why," not just the numbers) live in `NOTES.md`; this notebook is the executable record of exactly what was run.

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.DataStructs import ConvertToNumpyArray


# Unchanged from Week 2: strips counter-ions/solvents from a salt, keeping
# only the largest fragment.
def largest_fragment(mol):
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) == 1:
        return mol
    return max(frags, key=lambda m: m.GetNumHeavyAtoms())


# Unchanged from Week 2: 2048-bit Morgan fingerprint, radius 2.
def compute_morgan_fp(mol, radius=2, n_bits=2048):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=int)
    ConvertToNumpyArray(fp, arr)
    return arr

In [2]:
# Week 3 change: adds TPSA as a 6th descriptor (2054 feature columns total,
# up from Week 2's 2053) — everything else identical to Week 2's featurize().
def featurize(df, smiles_col='Drug', radius=2, n_bits=2048):
    fp_rows = []
    desc_rows = []
    for smi in df[smiles_col]:
        mol = Chem.MolFromSmiles(smi)
        mol = largest_fragment(mol)
        fp_rows.append(compute_morgan_fp(mol, radius=radius, n_bits=n_bits))
        desc_rows.append({
            'MolWt': Descriptors.MolWt(mol),
            'MolLogP': Descriptors.MolLogP(mol),
            'TPSA': Descriptors.TPSA(mol),
            'NumHDonors': Descriptors.NumHDonors(mol),
            'NumHAcceptors': Descriptors.NumHAcceptors(mol),
            'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        })

    fp_cols = [f'fp_{i}' for i in range(n_bits)]
    fp_df = pd.DataFrame(np.vstack(fp_rows), columns=fp_cols, index=df.index)
    desc_df = pd.DataFrame(desc_rows, index=df.index)
    return pd.concat([fp_df, desc_df], axis=1)

In [3]:
# Unchanged from Week 2: drops SMILES with conflicting labels, dedupes exact
# repeats. Used for classification targets (BBB, hERG, CYP3A4).
def dedupe_labels(df, smiles_col='Drug', y_col='Y'):
    label_counts = df.groupby(smiles_col)[y_col].nunique()
    conflicting = label_counts[label_counts > 1].index
    clean = df[~df[smiles_col].isin(conflicting)]
    return clean.drop_duplicates(subset=smiles_col)


# Unchanged from Week 2: averages Y across duplicate SMILES instead of
# dropping conflicts, appropriate for a continuous target. Used for Clearance.
def average_duplicate_targets(df, smiles_col='Drug', y_col='Y'):
    other_cols = [c for c in df.columns if c not in (smiles_col, y_col)]
    agg = {c: 'first' for c in other_cols}
    agg[y_col] = 'mean'
    return df.groupby(smiles_col, as_index=False).agg(agg)


# Unchanged from Week 2: scaffold split + a swappable dedup_fn.
def load_scaffold_split(dataset_class, name, dedup_fn=dedupe_labels):
    data = dataset_class(name=name)
    split = data.get_split(method='scaffold')
    train_df = dedup_fn(split['train'])
    test_df = dedup_fn(split['test'])
    return train_df, test_df

In [4]:
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    roc_auc_score, f1_score, balanced_accuracy_score,
)


# Unchanged from Week 2.
def evaluate_regression(model, X_test, y_test):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"RMSE: {rmse:.3f}")
    print(f"MAE:  {mae:.3f}")
    print(f"R^2:  {r2:.3f}")
    return {'rmse': rmse, 'mae': mae, 'r2': r2}


# Unchanged from Week 2.
def evaluate_classification(model, X_test, y_test):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    roc_auc = roc_auc_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    print(f"ROC-AUC:           {roc_auc:.3f}")
    print(f"F1:                {f1:.3f}")
    print(f"Balanced accuracy: {bal_acc:.3f}")
    return {'roc_auc': roc_auc, 'f1': f1, 'balanced_accuracy': bal_acc}

In [5]:
# Week 2's frozen baseline numbers (from Featurization_Baseline.ipynb /
# CLAUDE.md), used purely as a fixed reference to compare each Week 3 result
# against — not recomputed here, so Week 2 stays the stable "before" snapshot.
WEEK2_BASELINE = {
    'Solubility': {'rmse': 1.286, 'mae': 0.926, 'r2': 0.686},
    'BBB': {'roc_auc': 0.917, 'f1': 0.934, 'balanced_accuracy': 0.767},
    'hERG': {'roc_auc': 0.851, 'f1': 0.910, 'balanced_accuracy': 0.739},
    'CYP3A4': {'roc_auc': 0.879, 'f1': 0.750, 'balanced_accuracy': 0.779, 'auprc': 0.852},
    'Clearance': {'rmse': 44.480, 'mae': 35.069, 'r2': 0.115, 'spearman': 0.362},
}


# Prints each Week 2 metric next to the same metric computed this session
# (Week 3, +TPSA), with the change — the point of keeping Week 2 untouched.
def compare_to_week2(name, week3_metrics):
    print(f"\n--- {name}: Week 2 baseline -> Week 3 (+TPSA) ---")
    for metric, v2 in WEEK2_BASELINE[name].items():
        v3 = week3_metrics.get(metric)
        if v3 is None:
            continue
        print(f"{metric:>18}: {v2:.3f} -> {v3:.3f}  ({v3 - v2:+.3f})")

## Solubility (+TPSA)

In [6]:
from tdc.single_pred import ADME
from sklearn.ensemble import RandomForestRegressor

train_df, test_df = load_scaffold_split(ADME, 'Solubility_AqSolDB')
X_train = featurize(train_df)
X_test = featurize(test_df)
y_train = train_df['Y'].values
y_test = test_df['Y'].values

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

metrics = evaluate_regression(rf, X_test, y_test)
compare_to_week2('Solubility', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/9982 [00:00<?, ?it/s]

 10%|█         | 1007/9982 [00:00<00:00, 10043.72it/s]

...

 60%|██████    | 6018/9982 [00:00<00:00, 12286.25it/s]

 75%|███████▍  | 7476/9982 [00:00<00:00, 13028.03it/s]

 88%|████████▊ | 8812/9982 [00:00<00:00, 12576.76it/s]

100%|██████████| 9982/9982 [00:00<00:00, 11565.48it/s]

RMSE: 1.280
MAE:  0.920
R^2:  0.689

--- Solubility: Week 2 baseline -> Week 3 (+TPSA) ---
              rmse: 1.286 -> 1.280  (-0.006)
               mae: 0.926 -> 0.920  (-0.006)
                r2: 0.686 -> 0.689  (+0.003)


## BBB Permeability (+TPSA)

Expected biggest impact of the 5 — TPSA was the single cleanest class-separator found anywhere in Week 1 EDA for this exact property.

In [7]:
from sklearn.ensemble import RandomForestClassifier

bbb_train_df, bbb_test_df = load_scaffold_split(ADME, 'BBB_Martins')
X_bbb_train = featurize(bbb_train_df)
X_bbb_test = featurize(bbb_test_df)
y_bbb_train = bbb_train_df['Y'].values
y_bbb_test = bbb_test_df['Y'].values

rf_bbb = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_bbb.fit(X_bbb_train, y_bbb_train)

metrics = evaluate_classification(rf_bbb, X_bbb_test, y_bbb_test)
compare_to_week2('BBB', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/2030 [00:00<?, ?it/s]

 35%|███▌      | 719/2030 [00:00<00:00, 7182.68it/s]

 71%|███████   | 1438/2030 [00:00<00:00, 6638.23it/s]

100%|██████████| 2030/2030 [00:00<00:00, 7113.90it/s]

ROC-AUC:           0.910
F1:                0.936
Balanced accuracy: 0.779

--- BBB: Week 2 baseline -> Week 3 (+TPSA) ---
           roc_auc: 0.917 -> 0.910  (-0.007)
                f1: 0.934 -> 0.936  (+0.002)
 balanced_accuracy: 0.767 -> 0.779  (+0.012)


## hERG Blockade (+TPSA)

In [8]:
from tdc.single_pred import Tox

herg_train_df, herg_test_df = load_scaffold_split(Tox, 'hERG')
X_herg_train = featurize(herg_train_df)
X_herg_test = featurize(herg_test_df)
y_herg_train = herg_train_df['Y'].values
y_herg_test = herg_test_df['Y'].values

rf_herg = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_herg.fit(X_herg_train, y_herg_train)

metrics = evaluate_classification(rf_herg, X_herg_test, y_herg_test)
compare_to_week2('hERG', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/655 [00:00<?, ?it/s]

100%|██████████| 655/655 [00:00<00:00, 7091.74it/s]

ROC-AUC:           0.844
F1:                0.897
Balanced accuracy: 0.692

--- hERG: Week 2 baseline -> Week 3 (+TPSA) ---
           roc_auc: 0.851 -> 0.844  (-0.007)
                f1: 0.910 -> 0.897  (-0.013)
 balanced_accuracy: 0.739 -> 0.692  (-0.047)


## CYP3A4 Inhibition (+TPSA)

Week 1 EDA found essentially no separation from TPSA for this property specifically — a useful contrast case against BBB's expected-strong effect.

In [9]:
from sklearn.metrics import average_precision_score

cyp3a4_train_df, cyp3a4_test_df = load_scaffold_split(ADME, 'CYP3A4_Veith')
X_cyp3a4_train = featurize(cyp3a4_train_df)
X_cyp3a4_test = featurize(cyp3a4_test_df)
y_cyp3a4_train = cyp3a4_train_df['Y'].values
y_cyp3a4_test = cyp3a4_test_df['Y'].values

rf_cyp3a4 = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_cyp3a4.fit(X_cyp3a4_train, y_cyp3a4_train)

metrics = evaluate_classification(rf_cyp3a4, X_cyp3a4_test, y_cyp3a4_test)
y_cyp3a4_proba = rf_cyp3a4.predict_proba(X_cyp3a4_test)[:, 1]
metrics['auprc'] = average_precision_score(y_cyp3a4_test, y_cyp3a4_proba)
print(f"AUPRC:             {metrics['auprc']:.3f}")
compare_to_week2('CYP3A4', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/12328 [00:00<?, ?it/s]

  4%|▍         | 525/12328 [00:00<00:02, 5241.67it/s]

...

 83%|████████▎ | 10174/12328 [00:01<00:00, 7558.03it/s]

 89%|████████▉ | 10977/12328 [00:01<00:00, 7694.70it/s]

 95%|█████████▌| 11767/12328 [00:01<00:00, 7750.89it/s]

100%|██████████| 12328/12328 [00:01<00:00, 7196.33it/s]

ROC-AUC:           0.878
F1:                0.749
Balanced accuracy: 0.779
AUPRC:             0.851

--- CYP3A4: Week 2 baseline -> Week 3 (+TPSA) ---
           roc_auc: 0.879 -> 0.878  (-0.001)
                f1: 0.750 -> 0.749  (-0.001)
 balanced_accuracy: 0.779 -> 0.779  (-0.000)
             auprc: 0.852 -> 0.851  (-0.001)


## Clearance (+TPSA)

Uses `average_duplicate_targets` for the same reason as Week 2 (continuous target, real duplicate measurements — see NOTES.md). The assay-censoring issue (Y pinned at 3.0/150.0) is still not addressed here, unchanged from Week 2.

In [10]:
from scipy.stats import spearmanr

clearance_train_df, clearance_test_df = load_scaffold_split(
    ADME, 'Clearance_Hepatocyte_AZ', dedup_fn=average_duplicate_targets
)
X_clearance_train = featurize(clearance_train_df)
X_clearance_test = featurize(clearance_test_df)
y_clearance_train = clearance_train_df['Y'].values
y_clearance_test = clearance_test_df['Y'].values

rf_clearance = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_clearance.fit(X_clearance_train, y_clearance_train)

metrics = evaluate_regression(rf_clearance, X_clearance_test, y_clearance_test)
y_clearance_pred = rf_clearance.predict(X_clearance_test)
metrics['spearman'], _ = spearmanr(y_clearance_test, y_clearance_pred)
print(f"Spearman:          {metrics['spearman']:.3f}")
compare_to_week2('Clearance', metrics)

Found local copy...


Loading...


Done!


  0%|          | 0/1213 [00:00<?, ?it/s]

 53%|█████▎    | 642/1213 [00:00<00:00, 6395.18it/s]

100%|██████████| 1213/1213 [00:00<00:00, 6391.27it/s]

RMSE: 44.119
MAE:  34.950
R^2:  0.129
Spearman:          0.391

--- Clearance: Week 2 baseline -> Week 3 (+TPSA) ---
              rmse: 44.480 -> 44.119  (-0.361)
               mae: 35.069 -> 34.950  (-0.119)
                r2: 0.115 -> 0.129  (+0.014)
          spearman: 0.362 -> 0.391  (+0.029)


## Digging deeper: where does TPSA actually rank, and is hERG's drop real or noise?

Two concrete checks rather than just theorizing about the results above:

1. **TPSA's feature-importance rank** in each already-trained model, out of all 2054 features — tests the "TPSA may be redundant with fingerprint bits" hypothesis directly. If it ranks low despite being a strong *univariate* EDA signal, that supports redundancy with the fingerprint.
2. **hERG specifically:** since its test set is only 125 molecules, retrain the identical model with TPSA dropped (reusing the already-computed features, not recomputing anything) and check how many individual predictions actually flip, and whether those flips were close calls (probability near 0.5) or confident predictions changing — a few close calls flipping would support "small-test-set noise," not a real regression.

In [11]:
# TPSA's importance rank in each fitted model (1 = most important of 2054).
fitted_models = {
    'Solubility': (rf, X_train),
    'BBB': (rf_bbb, X_bbb_train),
    'hERG': (rf_herg, X_herg_train),
    'CYP3A4': (rf_cyp3a4, X_cyp3a4_train),
    'Clearance': (rf_clearance, X_clearance_train),
}

for name, (model, X_tr) in fitted_models.items():
    importances = pd.Series(model.feature_importances_, index=X_tr.columns)
    ranked = importances.sort_values(ascending=False)
    tpsa_rank = ranked.index.get_loc('TPSA') + 1
    tpsa_importance = importances['TPSA']
    percentile = 100 * (1 - tpsa_rank / len(ranked))
    print(f"{name:>10}: TPSA importance={tpsa_importance:.5f}, rank {tpsa_rank}/{len(ranked)} (top {100-percentile:.1f}%)")

Solubility: TPSA importance=0.03507, rank 3/2054 (top 0.1%)
       BBB: TPSA importance=0.03721, rank 2/2054 (top 0.1%)
      hERG: TPSA importance=0.02463, rank 3/2054 (top 0.1%)
    CYP3A4: TPSA importance=0.01488, rank 5/2054 (top 0.2%)
 Clearance: TPSA importance=0.07305, rank 1/2054 (top 0.0%)


In [12]:
# hERG without TPSA (reusing the already-computed features, just dropping
# the one column) — identical config otherwise, so any prediction difference
# is attributable to that one column alone, not random-seed variation.
X_herg_train_notpsa = X_herg_train.drop(columns='TPSA')
X_herg_test_notpsa = X_herg_test.drop(columns='TPSA')

rf_herg_notpsa = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf_herg_notpsa.fit(X_herg_train_notpsa, y_herg_train)

pred_with_tpsa = rf_herg.predict(X_herg_test)
pred_without_tpsa = rf_herg_notpsa.predict(X_herg_test_notpsa)
proba_with_tpsa = rf_herg.predict_proba(X_herg_test)[:, 1]
proba_without_tpsa = rf_herg_notpsa.predict_proba(X_herg_test_notpsa)[:, 1]

flipped = pred_with_tpsa != pred_without_tpsa
print(f"Test set size: {len(y_herg_test)}")
print(f"Predictions that flipped when TPSA was added: {flipped.sum()}")
print(f"Of those, probability (without TPSA) for each flipped case:")
print(np.round(proba_without_tpsa[flipped], 3))
print(f"Of those, probability (with TPSA) for each flipped case:")
print(np.round(proba_with_tpsa[flipped], 3))

Test set size: 125
Predictions that flipped when TPSA was added: 3
Of those, probability (without TPSA) for each flipped case:
[0.492 0.48  0.493]
Of those, probability (with TPSA) for each flipped case:
[0.505 0.54  0.547]


## Experiment 2: Gradient Boosting (XGBoost) vs. Random Forest

Random Forest builds many trees independently and averages them. Gradient boosting builds trees sequentially — each new tree specifically targets the *errors* (residuals) left by the trees before it, rather than being trained independently. XGBoost is the classical, well-documented implementation of this, and it's what many of the strongest classical-ML entries on the TDC leaderboards are built on.

Testing on 2 properties first (Solubility — regression, BBB — classification) rather than all 5, per the plan's own Week 3 framing: this is a comparison to learn from, not a full re-run of the panel.

Reuses the exact same TPSA-augmented features already computed earlier in this notebook (`X_train`/`X_test` for Solubility, `X_bbb_train`/`X_bbb_test` for BBB) — only the model changes, so this isolates "does the model family matter" as its own question, separate from Experiment 1's "does the feature set matter."

Compared against **two** references: Week 2's frozen RF baseline (no TPSA), and this notebook's own Experiment 1 RF+TPSA result — so it's possible to tell whether any improvement comes from XGBoost itself, or is just carried over from adding TPSA.

In [13]:
# This experiment's own RF+TPSA results (from earlier in this notebook),
# kept as a second reference point alongside WEEK2_BASELINE.
WEEK3_RF_TPSA = {
    'Solubility': {'rmse': 1.280, 'mae': 0.920, 'r2': 0.689},
    'BBB': {'roc_auc': 0.910, 'f1': 0.936, 'balanced_accuracy': 0.779},
}


# Generic version of compare_to_week2 — prints any named metrics next to a
# given reference dict, so the same helper works for either comparison.
def compare_metrics(label, reference, new_metrics):
    print(f"\n--- {label} ---")
    for metric, v_old in reference.items():
        v_new = new_metrics.get(metric)
        if v_new is None:
            continue
        print(f"{metric:>18}: {v_old:.3f} -> {v_new:.3f}  ({v_new - v_old:+.3f})")

### Solubility — XGBoost

In [14]:
from xgboost import XGBRegressor

# n_estimators matched to the RF baseline (200) for a like-for-like
# comparison; no tuning yet (that's a separate Week 3 step).
xgb_sol = XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1)
xgb_sol.fit(X_train, y_train)

xgb_sol_metrics = evaluate_regression(xgb_sol, X_test, y_test)
compare_metrics('Solubility: Week 2 RF (no TPSA) -> XGBoost (+TPSA)', WEEK2_BASELINE['Solubility'], xgb_sol_metrics)
compare_metrics('Solubility: Week 3 RF+TPSA -> XGBoost (+TPSA)', WEEK3_RF_TPSA['Solubility'], xgb_sol_metrics)

RMSE: 1.264
MAE:  0.932
R^2:  0.696

--- Solubility: Week 2 RF (no TPSA) -> XGBoost (+TPSA) ---
              rmse: 1.286 -> 1.264  (-0.022)
               mae: 0.926 -> 0.932  (+0.006)
                r2: 0.686 -> 0.696  (+0.010)

--- Solubility: Week 3 RF+TPSA -> XGBoost (+TPSA) ---
              rmse: 1.280 -> 1.264  (-0.016)
               mae: 0.920 -> 0.932  (+0.012)
                r2: 0.689 -> 0.696  (+0.007)


### BBB — XGBoost

In [15]:
from xgboost import XGBClassifier

# XGBoost doesn't have class_weight='balanced' like sklearn's RF — the
# equivalent is scale_pos_weight = (count of negative class) / (count of
# positive class), computed here rather than hardcoded so it matches
# whatever the actual train split's imbalance is.
scale_pos_weight = (y_bbb_train == 0).sum() / (y_bbb_train == 1).sum()

xgb_bbb = XGBClassifier(
    n_estimators=200, scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1
)
xgb_bbb.fit(X_bbb_train, y_bbb_train)

xgb_bbb_metrics = evaluate_classification(xgb_bbb, X_bbb_test, y_bbb_test)
compare_metrics('BBB: Week 2 RF (no TPSA) -> XGBoost (+TPSA)', WEEK2_BASELINE['BBB'], xgb_bbb_metrics)
compare_metrics('BBB: Week 3 RF+TPSA -> XGBoost (+TPSA)', WEEK3_RF_TPSA['BBB'], xgb_bbb_metrics)

ROC-AUC:           0.907
F1:                0.921
Balanced accuracy: 0.823

--- BBB: Week 2 RF (no TPSA) -> XGBoost (+TPSA) ---
           roc_auc: 0.917 -> 0.907  (-0.010)
                f1: 0.934 -> 0.921  (-0.013)
 balanced_accuracy: 0.767 -> 0.823  (+0.056)

--- BBB: Week 3 RF+TPSA -> XGBoost (+TPSA) ---
           roc_auc: 0.910 -> 0.907  (-0.003)
                f1: 0.936 -> 0.921  (-0.015)
 balanced_accuracy: 0.779 -> 0.823  (+0.044)


## Experiment 3: Hyperparameter Tuning (RF + XGBoost, Solubility + BBB)

Tunes both model families already built (RF, XGBoost) on the same 2 properties used for Experiment 2, via `RandomizedSearchCV` over a small parameter space — matching the project plan's "basic hyperparameter tuning... using cross-validation on train only."

**CV uses scaffold-grouped folds (`GroupKFold`), not plain K-fold.** Plain K-fold would let molecules that share a scaffold end up split across different folds — a milder version of the exact leakage risk scaffold splitting was introduced to prevent in the first place (Week 2 note: "scaffold split... tests generalization to novel chemical space"). Grouping CV folds by scaffold keeps every fold's internal validation as realistic as the actual train/test split, at the cost of a bit more code.

Scored on each property's actual leaderboard metric (MAE for Solubility, ROC-AUC for BBB) so the search optimizes for what's actually being compared against, not a generic default.

Each tuned model is compared against: Week 2's frozen baseline, this notebook's RF+TPSA result, and the untuned XGBoost result from Experiment 2.

In [16]:
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import RandomizedSearchCV, GroupKFold


# Assigns each molecule a scaffold-group label (its Bemis-Murcko scaffold,
# the same core-structure concept TDC's own scaffold split is built on), so
# GroupKFold can keep every scaffold's molecules together in one CV fold
# instead of splitting them across folds.
def compute_scaffold_groups(df, smiles_col='Drug'):
    groups = []
    for smi in df[smiles_col]:
        mol = Chem.MolFromSmiles(smi)
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        groups.append(scaffold)
    return groups

In [17]:
# Explicit callable scorer for XGBClassifier + ROC-AUC, needed because of a
# real version-compatibility bug: scikit-learn 1.7.2 (pulled in as a
# dependency when xgboost was installed) doesn't correctly recognize this
# XGBoost version's classifier via its newer auto-detection machinery for
# the 'roc_auc' string scorer, silently returning NaN for every CV fold
# instead of raising — confirmed directly with error_score='raise', which
# surfaced "Got a regressor with response_method=predict_proba instead."
# Passing a callable scorer that calls predict_proba directly bypasses that
# broken detection. RandomForestClassifier is unaffected (only used here for
# the XGBoost tuning searches below).
def proba_roc_auc(estimator, X, y):
    proba = estimator.predict_proba(X)[:, 1]
    return roc_auc_score(y, proba)

### Solubility — tuned Random Forest

In [18]:
sol_groups = compute_scaffold_groups(train_df)

# Small parameter space, per the plan's "basic hyperparameter tuning" scope —
# not an exhaustive grid. n_iter=15 candidates x 5 folds = 75 fits.
rf_param_grid = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.3],
}

rf_sol_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=rf_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
)
rf_sol_search.fit(X_train, y_train, groups=sol_groups)

print(f"Best CV MAE: {-rf_sol_search.best_score_:.3f}")
print(f"Best params: {rf_sol_search.best_params_}")

rf_sol_tuned_metrics = evaluate_regression(rf_sol_search.best_estimator_, X_test, y_test)
compare_metrics('Solubility: Week 2 RF -> tuned RF', WEEK2_BASELINE['Solubility'], rf_sol_tuned_metrics)
compare_metrics('Solubility: Week 3 RF+TPSA -> tuned RF', WEEK3_RF_TPSA['Solubility'], rf_sol_tuned_metrics)
compare_metrics('Solubility: untuned XGBoost -> tuned RF', xgb_sol_metrics, rf_sol_tuned_metrics)

/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from 

/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


Best CV MAE: 0.881
Best params: {'n_estimators': 200, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 30}
RMSE: 1.256
MAE:  0.912
R^2:  0.700

--- Solubility: Week 2 RF -> tuned RF ---
              rmse: 1.286 -> 1.256  (-0.030)
               mae: 0.926 -> 0.912  (-0.014)
                r2: 0.686 -> 0.700  (+0.014)

--- Solubility: Week 3 RF+TPSA -> tuned RF ---
              rmse: 1.280 -> 1.256  (-0.024)
               mae: 0.920 -> 0.912  (-0.008)
                r2: 0.689 -> 0.700  (+0.011)

--- Solubility: untuned XGBoost -> tuned RF ---
              rmse: 1.264 -> 1.256  (-0.008)
               mae: 0.932 -> 0.912  (-0.020)
                r2: 0.696 -> 0.700  (+0.004)


### Solubility — tuned XGBoost

In [19]:
xgb_param_grid = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
}

xgb_sol_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_distributions=xgb_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
)
xgb_sol_search.fit(X_train, y_train, groups=sol_groups)

print(f"Best CV MAE: {-xgb_sol_search.best_score_:.3f}")
print(f"Best params: {xgb_sol_search.best_params_}")

xgb_sol_tuned_metrics = evaluate_regression(xgb_sol_search.best_estimator_, X_test, y_test)
compare_metrics('Solubility: Week 2 RF -> tuned XGBoost', WEEK2_BASELINE['Solubility'], xgb_sol_tuned_metrics)
compare_metrics('Solubility: untuned XGBoost -> tuned XGBoost', xgb_sol_metrics, xgb_sol_tuned_metrics)
compare_metrics('Solubility: tuned RF -> tuned XGBoost', rf_sol_tuned_metrics, xgb_sol_tuned_metrics)

Best CV MAE: 0.877
Best params: {'subsample': 1.0, 'n_estimators': 200, 'max_depth': 9, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
RMSE: 1.235
MAE:  0.899
R^2:  0.710

--- Solubility: Week 2 RF -> tuned XGBoost ---
              rmse: 1.286 -> 1.235  (-0.051)
               mae: 0.926 -> 0.899  (-0.027)
                r2: 0.686 -> 0.710  (+0.024)

--- Solubility: untuned XGBoost -> tuned XGBoost ---
              rmse: 1.264 -> 1.235  (-0.029)
               mae: 0.932 -> 0.899  (-0.033)
                r2: 0.696 -> 0.710  (+0.014)

--- Solubility: tuned RF -> tuned XGBoost ---
              rmse: 1.256 -> 1.235  (-0.021)
               mae: 0.912 -> 0.899  (-0.013)
                r2: 0.700 -> 0.710  (+0.010)


### BBB — tuned Random Forest

In [20]:
bbb_groups = compute_scaffold_groups(bbb_train_df)

# class_weight='balanced' held fixed (not searched) so tuning targets genuine
# hyperparameters, not the imbalance-handling choice already made in Week 2.
rf_bbb_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42),
    param_distributions=rf_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
)
rf_bbb_search.fit(X_bbb_train, y_bbb_train, groups=bbb_groups)

print(f"Best CV ROC-AUC: {rf_bbb_search.best_score_:.3f}")
print(f"Best params: {rf_bbb_search.best_params_}")

rf_bbb_tuned_metrics = evaluate_classification(rf_bbb_search.best_estimator_, X_bbb_test, y_bbb_test)
compare_metrics('BBB: Week 2 RF -> tuned RF', WEEK2_BASELINE['BBB'], rf_bbb_tuned_metrics)
compare_metrics('BBB: Week 3 RF+TPSA -> tuned RF', WEEK3_RF_TPSA['BBB'], rf_bbb_tuned_metrics)
compare_metrics('BBB: untuned XGBoost -> tuned RF', xgb_bbb_metrics, rf_bbb_tuned_metrics)

/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from 

Best CV ROC-AUC: 0.910
Best params: {'n_estimators': 300, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20}
ROC-AUC:           0.915
F1:                0.925
Balanced accuracy: 0.794

--- BBB: Week 2 RF -> tuned RF ---
           roc_auc: 0.917 -> 0.915  (-0.002)
                f1: 0.934 -> 0.925  (-0.009)
 balanced_accuracy: 0.767 -> 0.794  (+0.027)

--- BBB: Week 3 RF+TPSA -> tuned RF ---
           roc_auc: 0.910 -> 0.915  (+0.005)
                f1: 0.936 -> 0.925  (-0.011)
 balanced_accuracy: 0.779 -> 0.794  (+0.015)

--- BBB: untuned XGBoost -> tuned RF ---
           roc_auc: 0.907 -> 0.915  (+0.007)
                f1: 0.921 -> 0.925  (+0.003)
 balanced_accuracy: 0.823 -> 0.794  (-0.028)


### BBB — tuned XGBoost

In [21]:
# scale_pos_weight held fixed (computed the same way as Experiment 2), same
# reasoning as class_weight='balanced' above. Uses the explicit proba_roc_auc
# scorer (not the 'roc_auc' string) — see the note above for why.
xgb_bbb_search = RandomizedSearchCV(
    estimator=XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42),
    param_distributions=xgb_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring=proba_roc_auc,
    random_state=42,
    n_jobs=-1,
)
xgb_bbb_search.fit(X_bbb_train, y_bbb_train, groups=bbb_groups)

print(f"Best CV ROC-AUC: {xgb_bbb_search.best_score_:.3f}")
print(f"Best params: {xgb_bbb_search.best_params_}")

xgb_bbb_tuned_metrics = evaluate_classification(xgb_bbb_search.best_estimator_, X_bbb_test, y_bbb_test)
compare_metrics('BBB: Week 2 RF -> tuned XGBoost', WEEK2_BASELINE['BBB'], xgb_bbb_tuned_metrics)
compare_metrics('BBB: untuned XGBoost -> tuned XGBoost', xgb_bbb_metrics, xgb_bbb_tuned_metrics)
compare_metrics('BBB: tuned RF -> tuned XGBoost', rf_bbb_tuned_metrics, xgb_bbb_tuned_metrics)

/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


Best CV ROC-AUC: 0.906
Best params: {'subsample': 0.85, 'n_estimators': 400, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.5}
ROC-AUC:           0.920
F1:                0.924
Balanced accuracy: 0.836

--- BBB: Week 2 RF -> tuned XGBoost ---
           roc_auc: 0.917 -> 0.920  (+0.003)
                f1: 0.934 -> 0.924  (-0.010)
 balanced_accuracy: 0.767 -> 0.836  (+0.069)

--- BBB: untuned XGBoost -> tuned XGBoost ---
           roc_auc: 0.907 -> 0.920  (+0.012)
                f1: 0.921 -> 0.924  (+0.003)
 balanced_accuracy: 0.823 -> 0.836  (+0.013)

--- BBB: tuned RF -> tuned XGBoost ---
           roc_auc: 0.915 -> 0.920  (+0.005)
                f1: 0.925 -> 0.924  (-0.000)
 balanced_accuracy: 0.794 -> 0.836  (+0.042)


## Experiment 4: Permutation Importance (Solubility + BBB)

Experiment 1 found that RF's default feature importance (MDI/Gini) is biased toward continuous features over binary ones — TPSA ranked top-5 in every model regardless of whether it actually moved the test score. **Permutation importance fixes this**, by measuring something concrete instead of a tree-structure artifact: take the already-trained model, shuffle one feature's values across the test set (breaking that feature's link to the target while leaving everything else intact), and see how much the model's real test score drops. A feature that matters a lot causes a big drop when scrambled; a feature the model ignores causes almost none — this works identically for continuous and binary features, unlike MDI.

**Scope, and why:** with 2048 individual fingerprint bits, permuting each one separately would be expensive and mostly uninterpretable (no single bit means anything on its own to a chemist). Instead: permute each of the 6 human-interpretable descriptors individually, and permute the **entire fingerprint as one block** (shuffling all 2048 columns together, row-wise, so each molecule gets some other molecule's whole fingerprint) to see how much the model relies on structural detail as a whole versus the 6 descriptors. This directly answers the plan's actual question — "which descriptors matter most" — without 2048 meaningless individual numbers.

**Models used:** the tuned XGBoost models from Experiment 3 — the best model found for each property, and the one that would actually be reported/defended, rather than re-analyzing an earlier, worse baseline.

Scored on each property's actual leaderboard metric (MAE for Solubility, ROC-AUC for BBB) so "importance" means "how much this feature affects the number that's actually being compared to the leaderboard."

In [22]:
# Custom grouped permutation importance: for each named group of columns
# (a single descriptor, or the whole 2048-bit fingerprint block), shuffle
# those columns' values across test-set rows (same shuffle applied jointly
# across every column in the group, so a multi-column group's internal
# structure survives — only its link to y is broken), rescore, and repeat
# n_repeats times to average out shuffle-to-shuffle noise. scoring_fn must
# return a score where higher = better, so "importance" = baseline - permuted
# is positive when shuffling hurt performance (i.e. the group mattered).
def permutation_importance_grouped(model, X_test, y_test, groups, scoring_fn, n_repeats=10, random_state=42):
    rng = np.random.RandomState(random_state)
    baseline = scoring_fn(model, X_test, y_test)
    results = {}
    for name, cols in groups.items():
        drops = []
        for _ in range(n_repeats):
            X_shuffled = X_test.copy()
            shuffled_order = rng.permutation(len(X_test))
            X_shuffled[cols] = X_test[cols].values[shuffled_order]
            score = scoring_fn(model, X_shuffled, y_test)
            drops.append(baseline - score)
        results[name] = {'importance_mean': np.mean(drops), 'importance_std': np.std(drops)}
    return baseline, results


# Prints groups ranked by importance, most important first.
def print_importance_ranking(baseline, results, metric_name):
    print(f"Baseline {metric_name}: {baseline:.4f}\n")
    ranked = sorted(results.items(), key=lambda kv: kv[1]['importance_mean'], reverse=True)
    for name, stats in ranked:
        print(f"{name:>28}: importance {stats['importance_mean']:+.4f} (± {stats['importance_std']:.4f})")

### Solubility — permutation importance (tuned XGBoost)

Week 1 EDA finding to check against: MolLogP showed the cleanest, strongest trend with solubility of any descriptor (more lipophilic = less soluble, well-established medicinal chemistry).

In [23]:
fp_cols = [c for c in X_test.columns if c.startswith('fp_')]
descriptor_cols = ['MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds']

sol_groups_for_importance = {'Fingerprint (all 2048 bits)': fp_cols}
sol_groups_for_importance.update({col: [col] for col in descriptor_cols})

# Negative MAE so higher = better, matching this dataset's leaderboard metric.
def neg_mae_scorer(model, X, y):
    return -mean_absolute_error(y, model.predict(X))

sol_baseline, sol_importance = permutation_importance_grouped(
    xgb_sol_search.best_estimator_, X_test, y_test, sol_groups_for_importance, neg_mae_scorer, n_repeats=10
)
print_importance_ranking(sol_baseline, sol_importance, 'neg MAE')

Baseline neg MAE: -0.8991

                     MolLogP: importance +0.8866 (± 0.0155)
 Fingerprint (all 2048 bits): importance +0.3137 (± 0.0118)
                       MolWt: importance +0.2346 (± 0.0126)
                        TPSA: importance +0.0604 (± 0.0034)
                  NumHDonors: importance +0.0183 (± 0.0029)
               NumHAcceptors: importance +0.0128 (± 0.0022)
           NumRotatableBonds: importance +0.0075 (± 0.0014)


### BBB — permutation importance (tuned XGBoost)

Week 1 EDA finding to check against: TPSA showed the cleanest class separation of any descriptor found anywhere in the whole panel (non-permeable ~110, permeable ~40-50) — independently reproducing the known "TPSA < ~90" rule of thumb for BBB penetration.

In [24]:
bbb_fp_cols = [c for c in X_bbb_test.columns if c.startswith('fp_')]
bbb_groups_for_importance = {'Fingerprint (all 2048 bits)': bbb_fp_cols}
bbb_groups_for_importance.update({col: [col] for col in descriptor_cols})

# ROC-AUC is already "higher = better", so it can be used directly as the
# scoring function (via predict_proba, matching how BBB is actually scored).
def roc_auc_scorer(model, X, y):
    return roc_auc_score(y, model.predict_proba(X)[:, 1])

bbb_baseline, bbb_importance = permutation_importance_grouped(
    xgb_bbb_search.best_estimator_, X_bbb_test, y_bbb_test, bbb_groups_for_importance, roc_auc_scorer, n_repeats=10
)
print_importance_ranking(bbb_baseline, bbb_importance, 'ROC-AUC')

Baseline ROC-AUC: 0.9196

 Fingerprint (all 2048 bits): importance +0.0988 (± 0.0151)
                        TPSA: importance +0.0354 (± 0.0093)
                  NumHDonors: importance +0.0212 (± 0.0062)
                       MolWt: importance +0.0089 (± 0.0042)
                     MolLogP: importance +0.0086 (± 0.0025)
               NumHAcceptors: importance +0.0029 (± 0.0012)
           NumRotatableBonds: importance +0.0009 (± 0.0027)


## Experiment 5: Failure Mode Analysis (Solubility + BBB)

Per the project plan's Week 3 step 4: "Document failure modes: where does the model perform poorly, and can you form a hypothesis why (e.g., underrepresented chemical space in scaffold split)?"

Rather than just eyeballing the worst predictions, we test that exact hypothesis directly: for each test molecule, compute its Tanimoto similarity (on the Morgan fingerprint) to its *nearest neighbor in the training set* — a proxy for "how novel is this chemotype to the model." Low similarity means the model has never seen anything structurally close to this molecule during training. If the hypothesis holds, error should be higher for low-similarity (novel) molecules. We also inspect the worst individual predictions by eye for any other pattern (e.g. the salt/inorganic outliers flagged back in Week 1 EDA). Uses the tuned XGBoost models from Experiment 3, same two properties as Experiment 4 for consistency.

In [25]:
# For each test molecule, the Tanimoto similarity to its nearest neighbor in
# the training set -- a proxy for "how novel is this chemotype", to test the
# project plan's suggested failure-mode hypothesis (underrepresented chemical
# space in the scaffold split). Computed via matrix ops on the fingerprint
# bits directly (Tanimoto = intersection / union of set bits), since a
# molecule-by-molecule pairwise loop would be far slower.
def nn_tanimoto_similarity(X_test_fp, X_train_fp):
    test_arr = X_test_fp.values.astype(np.float32)
    train_arr = X_train_fp.values.astype(np.float32)
    intersection = test_arr @ train_arr.T
    test_sums = test_arr.sum(axis=1, keepdims=True)
    train_sums = train_arr.sum(axis=1, keepdims=True)
    union = test_sums + train_sums.T - intersection
    tanimoto = intersection / np.where(union == 0, 1, union)
    return tanimoto.max(axis=1)

### Solubility — worst predictions + chemical-space hypothesis (tuned XGBoost)

In [26]:
sol_model = xgb_sol_search.best_estimator_
sol_pred = sol_model.predict(X_test)
sol_abs_err = np.abs(sol_pred - y_test)

sol_fail_df = pd.DataFrame({
    'SMILES': test_df['Drug'].values,
    'MolWt': X_test['MolWt'].values,
    'MolLogP': X_test['MolLogP'].values,
    'TPSA': X_test['TPSA'].values,
    'y_true': y_test,
    'y_pred': sol_pred,
    'abs_error': sol_abs_err,
})
sol_fail_df.sort_values('abs_error', ascending=False).head(15)

,SMILES,MolWt,MolLogP,TPSA,y_true,y_pred,abs_error
194,C1CO[Sb]2OCCO[Sb](O1)OCCO2,423.676,-0.91700,55.38,-9.024974,-2.346301,6.678673
1789,COCCOCCCn1c(=O)c2c(N)c3c(=O)c4ccccc4c(=O)c3c(N...,423.425,0.48160,143.71,-8.265049,-2.050026,6.215022
730,CC(=O)[O-].CCN(CC)c1ccc([C+](c2ccc(N(CC)CC)cc2...,456.698,7.24439,9.72,-0.139561,-6.330511,6.190950
107,ClC1=C(Cl)C2(Cl)C3CCC4C(CCC3C1(Cl)C2(Cl)Cl)C1(...,653.730,9.71620,0.00,-13.171900,-7.255924,5.915976
732,CC(=O)[O-].CCN(CC)c1ccc([C+](c2ccccc2)c2ccc(N(...,385.575,6.39819,6.48,-0.647988,-6.394611,5.746623
550,Clc1c(Cl)c(Cl)c(Oc2c(Cl)c(Cl)c(Cl)c(Cl)c2Cl)c(...,514.661,10.01290,9.23,-12.950000,-7.272559,5.677441
1576,CCCCC(CC)CNS(=O)(=O)c1ccc(N=Nc2c(C)c(C#N)c(=O)...,501.653,5.44420,136.91,-8.797313,-3.455811,5.341502
670,CCOC(=O)c1ccccc1-c1c2ccc(N(CC)CC)cc2[o+]c2cc(N...,471.621,7.40320,44.08,-1.280190,-6.593187,5.312997
186,N=C(N)N.O=C(O)C1=NN(c2ccc(S(=O)(=O)O)cc2)C(=O)...,468.425,1.11970,203.43,-7.751247,-2.448480,5.302767
1864,O=C([O-])c1ccccc1-c1c2cc(Br)c(=O)c(Br)c-2oc2c(...,645.879,5.05190,93.40,-0.840018,-6.098380,5.258362


In [27]:
from scipy.stats import spearmanr

sol_nn_sim = nn_tanimoto_similarity(X_test[fp_cols], X_train[fp_cols])

corr, pval = spearmanr(sol_nn_sim, sol_abs_err)
print(f"Spearman correlation (train-similarity vs. abs error): {corr:.3f} (p={pval:.2e})")

worst_10pct = sol_fail_df['abs_error'] >= sol_fail_df['abs_error'].quantile(0.90)
print(f"\nMean train-similarity, worst 10% of predictions: {sol_nn_sim[worst_10pct.values].mean():.3f}")
print(f"Mean train-similarity, remaining 90%:             {sol_nn_sim[~worst_10pct.values].mean():.3f}")

Spearman correlation (train-similarity vs. abs error): -0.064 (p=4.15e-03)

Mean train-similarity, worst 10% of predictions: 0.451
Mean train-similarity, remaining 90%:             0.462


### BBB — worst predictions + chemical-space hypothesis (tuned XGBoost)

In [28]:
bbb_model = xgb_bbb_search.best_estimator_
bbb_proba = bbb_model.predict_proba(X_bbb_test)[:, 1]
bbb_pred = (bbb_proba >= 0.5).astype(int)
bbb_correct = (bbb_pred == y_bbb_test)

# "Confidence error": distance from the correct answer -- 0 for a confident
# correct call, close to 1 for a confident wrong call.
bbb_conf_error = np.where(y_bbb_test == 1, 1 - bbb_proba, bbb_proba)

bbb_fail_df = pd.DataFrame({
    'SMILES': bbb_test_df['Drug'].values,
    'MolLogP': X_bbb_test['MolLogP'].values,
    'TPSA': X_bbb_test['TPSA'].values,
    'y_true': y_bbb_test,
    'predicted_proba': bbb_proba,
    'correct': bbb_correct,
    'confidence_error': bbb_conf_error,
})
bbb_fail_df.sort_values('confidence_error', ascending=False).head(15)

,SMILES,MolLogP,TPSA,y_true,predicted_proba,correct,confidence_error
153,CC[N+](C)(C)Cc1ccccc1Br.Cc1ccc(S(=O)(=O)[O-])cc1,3.04540,0.00,0,0.995467,False,0.995467
5,Cc1onc(-c2ccccc2Cl)c1C(=O)N[C@@H]1C(=O)N2[C@@H...,2.54872,112.74,1,0.005292,False,0.994708
73,CC(C[N+](C)(C)C)OC(N)=O.[Cl-],0.17640,52.32,0,0.982888,False,0.982888
273,CN(C)c1ccc(O)c2c1C[C@H]1C[C@H]3[C@H](N(C)C)C(O...,0.18660,164.63,1,0.032469,False,0.967531
263,CC[C@H](NC(=O)c1c(O)c(-c2ccccc2)nc2ccccc12)c1c...,5.48850,62.22,1,0.044608,False,0.955392
265,CC[C@H](NC(=O)c1c(OCCCN(C)C)c(-c2ccccc2)nc2ccc...,6.11340,54.46,1,0.055794,False,0.944206
299,O=C(Cc1ccc(Cl)c(Cl)c1)N1CCc2[nH]cnc2[C@H]1CN1C...,3.48080,52.23,0,0.942082,False,0.942082
7,C[S+](CC[C@H](N)C(=O)[O-])C[C@H]1O[C@@H](n2cnc...,-3.25690,185.46,1,0.058531,False,0.941469
88,O=S(=O)([O-])CCS.[Na+],-0.53860,57.20,0,0.940624,False,0.940624
150,CC(C)(C)NCC(O)COc1cccc2c1CCC(=O)N2,1.69910,70.59,0,0.924226,False,0.924226


In [29]:
bbb_nn_sim = nn_tanimoto_similarity(X_bbb_test[bbb_fp_cols], X_bbb_train[bbb_fp_cols])

corr, pval = spearmanr(bbb_nn_sim, bbb_conf_error)
print(f"Spearman correlation (train-similarity vs. confidence error): {corr:.3f} (p={pval:.2e})")

print(f"\nMean train-similarity, misclassified:        {bbb_nn_sim[~bbb_correct].mean():.3f}")
print(f"Mean train-similarity, correctly classified: {bbb_nn_sim[bbb_correct].mean():.3f}")

Spearman correlation (train-similarity vs. confidence error): -0.087 (p=8.42e-02)

Mean train-similarity, misclassified:        0.398
Mean train-similarity, correctly classified: 0.415


### Findings — the plan's hypothesis didn't hold; a sharper one did

**The chemical-space-novelty hypothesis (per the plan) is not supported.** For both properties, Spearman correlation between train-set similarity and error is essentially zero (Solubility: -0.064; BBB: -0.087, not even significant at p<0.05), and the worst-predicted molecules aren't meaningfully less similar to the training set than everything else (Solubility: 0.451 vs. 0.462 mean similarity; BBB: 0.398 vs. 0.415). The scaffold split's "novel chemotype" framing doesn't explain these failures — a negative result worth reporting honestly rather than forcing a fit.

**Looking at the actual worst-predicted molecules instead reveals a much sharper pattern:**

- **Solubility — the worst 15 are almost entirely industrial dyes (anthraquinone, triarylmethane/malachite-green-type, xanthene/eosin-type, azo) and heavily chlorinated pesticide-like compounds (mirex-like cages, decachlorodiphenyl ether)** — not drug-like molecules at all. AqSolDB compiles solubility data from multiple sources including environmental/industrial chemicals, not just drugs. This matters mechanistically: Experiment 4 showed this model relies overwhelmingly on MolLogP. For charged dye salts (many of the worst 15 are drawn as a cation + acetate/oxalate counterion), MolLogP computed on the organic fragment doesn't reflect the true solubility driver — these compounds dissolve (or don't) based on ionic dissociation and crystal packing, not simple octanol-water partitioning. The model's core assumption breaks down specifically for this compound class.
- **BBB — the worst errors cluster around formally charged species (quaternary ammonium salts, sulfonate/carboxylate salts) and high-TPSA molecules labeled permeable.** Quaternary ammonium compounds (e.g. a benzyl-ammonium tosylate salt) get predicted as highly BBB-permeable (TPSA≈0, looks "drug-like" to the model) despite a permanent positive charge that should exclude them biologically — the feature set doesn't encode formal charge state. Separately, several high-TPSA molecules (TPSA 130-170+, resembling tetracycline- and nucleoside-prodrug-type scaffolds) are labeled permeable but predicted confidently impermeable — these directly contradict the "TPSA<90" rule the model has learned, and are consistent with active transport (carrier-mediated uptake) getting a molecule across the BBB independent of passive-diffusion descriptors. No feature set built purely from static 2D structure can capture that mechanism — a textbook, known limitation of descriptor-based BBB models, not a bug.

**Same underlying lesson for both properties:** the model's descriptors (MolLogP, TPSA, 2D fingerprints) encode passive, static molecular structure. Both failure clusters are cases where the *real* driver is something structural descriptors don't see — ionic/salt behavior for Solubility, formal charge state and active transport for BBB.

## Extending Week 3 to hERG and CYP3A4

Applies the same treatment already done for Solubility + BBB to the remaining two classification properties (Clearance is deliberately held back until the assay-censoring question is decided). Experiment 1's TPSA addition already covered all 5 properties, so the RF+TPSA baseline for hERG and CYP3A4 already exists (`rf_herg`, `rf_cyp3a4`, computed earlier) — no need to redo that step. What's new here: XGBoost comparison, hyperparameter tuning, permutation importance, and failure mode analysis, reusing every helper function already built (`compare_metrics`, `compute_scaffold_groups`, `rf_param_grid`/`xgb_param_grid`, `permutation_importance_grouped`, `nn_tanimoto_similarity`) rather than redefining anything.

CYP3A4's leaderboard metric is AUPRC, not ROC-AUC (as established in Week 2) — so its tuning and permutation importance are scored on AUPRC specifically, the same way Solubility's used MAE and BBB's used ROC-AUC.

In [30]:
# Adds this notebook's own hERG/CYP3A4 RF+TPSA results (computed earlier,
# in the Experiment 1 section) to the existing reference dict, so the
# XGBoost/tuning comparisons below have the same two reference points
# (Week 2 baseline + this notebook's RF+TPSA) that Solubility/BBB got.
WEEK3_RF_TPSA['hERG'] = {'roc_auc': 0.844, 'f1': 0.897, 'balanced_accuracy': 0.692}
WEEK3_RF_TPSA['CYP3A4'] = {'roc_auc': 0.878, 'f1': 0.749, 'balanced_accuracy': 0.779, 'auprc': 0.851}

### hERG — XGBoost vs. Random Forest (RF+TPSA baseline)

In [31]:
scale_pos_weight_herg = (y_herg_train == 0).sum() / (y_herg_train == 1).sum()

xgb_herg = XGBClassifier(
    n_estimators=200, scale_pos_weight=scale_pos_weight_herg, random_state=42, n_jobs=-1
)
xgb_herg.fit(X_herg_train, y_herg_train)

xgb_herg_metrics = evaluate_classification(xgb_herg, X_herg_test, y_herg_test)
compare_metrics('hERG: Week 2 RF (no TPSA) -> XGBoost (+TPSA)', WEEK2_BASELINE['hERG'], xgb_herg_metrics)
compare_metrics('hERG: Week 3 RF+TPSA -> XGBoost (+TPSA)', WEEK3_RF_TPSA['hERG'], xgb_herg_metrics)

ROC-AUC:           0.800
F1:                0.884
Balanced accuracy: 0.748

--- hERG: Week 2 RF (no TPSA) -> XGBoost (+TPSA) ---
           roc_auc: 0.851 -> 0.800  (-0.051)
                f1: 0.910 -> 0.884  (-0.026)
 balanced_accuracy: 0.739 -> 0.748  (+0.009)

--- hERG: Week 3 RF+TPSA -> XGBoost (+TPSA) ---
           roc_auc: 0.844 -> 0.800  (-0.044)
                f1: 0.897 -> 0.884  (-0.013)
 balanced_accuracy: 0.692 -> 0.748  (+0.056)


### hERG — hyperparameter tuning (RF + XGBoost)

In [32]:
herg_groups = compute_scaffold_groups(herg_train_df)

rf_herg_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42),
    param_distributions=rf_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
)
rf_herg_search.fit(X_herg_train, y_herg_train, groups=herg_groups)

print(f"Best CV ROC-AUC: {rf_herg_search.best_score_:.3f}")
print(f"Best params: {rf_herg_search.best_params_}")

rf_herg_tuned_metrics = evaluate_classification(rf_herg_search.best_estimator_, X_herg_test, y_herg_test)
compare_metrics('hERG: Week 2 RF -> tuned RF', WEEK2_BASELINE['hERG'], rf_herg_tuned_metrics)

/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


Best CV ROC-AUC: 0.853
Best params: {'n_estimators': 300, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20}
ROC-AUC:           0.848
F1:                0.908
Balanced accuracy: 0.760

--- hERG: Week 2 RF -> tuned RF ---
           roc_auc: 0.851 -> 0.848  (-0.003)
                f1: 0.910 -> 0.908  (-0.002)
 balanced_accuracy: 0.739 -> 0.760  (+0.021)


In [33]:
xgb_herg_search = RandomizedSearchCV(
    estimator=XGBClassifier(scale_pos_weight=scale_pos_weight_herg, random_state=42),
    param_distributions=xgb_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring=proba_roc_auc,
    random_state=42,
    n_jobs=-1,
)
xgb_herg_search.fit(X_herg_train, y_herg_train, groups=herg_groups)

print(f"Best CV ROC-AUC: {xgb_herg_search.best_score_:.3f}")
print(f"Best params: {xgb_herg_search.best_params_}")

xgb_herg_tuned_metrics = evaluate_classification(xgb_herg_search.best_estimator_, X_herg_test, y_herg_test)
compare_metrics('hERG: Week 2 RF -> tuned XGBoost', WEEK2_BASELINE['hERG'], xgb_herg_tuned_metrics)
compare_metrics('hERG: tuned RF -> tuned XGBoost', rf_herg_tuned_metrics, xgb_herg_tuned_metrics)

Best CV ROC-AUC: 0.845
Best params: {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
ROC-AUC:           0.856
F1:                0.898
Balanced accuracy: 0.739

--- hERG: Week 2 RF -> tuned XGBoost ---
           roc_auc: 0.851 -> 0.856  (+0.005)
                f1: 0.910 -> 0.898  (-0.012)
 balanced_accuracy: 0.739 -> 0.739  (-0.000)

--- hERG: tuned RF -> tuned XGBoost ---
           roc_auc: 0.848 -> 0.856  (+0.008)
                f1: 0.908 -> 0.898  (-0.010)
 balanced_accuracy: 0.760 -> 0.739  (-0.021)


### hERG — permutation importance (tuned XGBoost)

In [34]:
herg_fp_cols = [c for c in X_herg_test.columns if c.startswith('fp_')]
herg_groups_for_importance = {'Fingerprint (all 2048 bits)': herg_fp_cols}
herg_groups_for_importance.update({col: [col] for col in descriptor_cols})

herg_baseline, herg_importance = permutation_importance_grouped(
    xgb_herg_search.best_estimator_, X_herg_test, y_herg_test, herg_groups_for_importance, roc_auc_scorer, n_repeats=10
)
print_importance_ranking(herg_baseline, herg_importance, 'ROC-AUC')

Baseline ROC-AUC: 0.8562

 Fingerprint (all 2048 bits): importance +0.0674 (± 0.0186)
                       MolWt: importance +0.0475 (± 0.0167)
                        TPSA: importance +0.0449 (± 0.0199)
           NumRotatableBonds: importance +0.0176 (± 0.0098)
                     MolLogP: importance +0.0160 (± 0.0192)
                  NumHDonors: importance +0.0069 (± 0.0017)
               NumHAcceptors: importance +0.0008 (± 0.0041)


### hERG — failure mode analysis

In [35]:
herg_model = xgb_herg_search.best_estimator_
herg_proba = herg_model.predict_proba(X_herg_test)[:, 1]
herg_pred = (herg_proba >= 0.5).astype(int)
herg_correct = (herg_pred == y_herg_test)

# "Confidence error": distance from the correct answer -- 0 for a confident
# correct call, close to 1 for a confident wrong call.
herg_conf_error = np.where(y_herg_test == 1, 1 - herg_proba, herg_proba)

herg_fail_df = pd.DataFrame({
    'SMILES': herg_test_df['Drug'].values,
    'MolLogP': X_herg_test['MolLogP'].values,
    'TPSA': X_herg_test['TPSA'].values,
    'y_true': y_herg_test,
    'predicted_proba': herg_proba,
    'correct': herg_correct,
    'confidence_error': herg_conf_error,
})
herg_fail_df.sort_values('confidence_error', ascending=False).head(15)

,SMILES,MolLogP,TPSA,y_true,predicted_proba,correct,confidence_error
38,CCCC(C)C1(CC)C(=O)NC(=S)NC1=O,1.34990,58.20,1.0,0.084220,False,0.915780
1,CC(=O)Nc1ccc(CN2CCC(NC(=O)c3cc(=O)c4ccc(F)cc4o...,3.28490,91.65,0.0,0.898373,False,0.898373
65,CC[C@H](O)c1cn(-c2ccc(F)cc2)c2ccc(Cl)cc12,4.86640,25.16,0.0,0.874769,False,0.874769
72,Fc1ccc(-n2cc(C3CC[NH2+]CC3)c3cc(Cl)ccc32)cc1,3.86380,21.54,0.0,0.862018,False,0.862018
41,O=C1CCC(c2ccc(Nc3ccncc3)cc2)=NN1,2.43930,66.38,1.0,0.147455,False,0.852545
48,C[NH+](C)CCO[C@](C)(c1ccccc1)c1ccccn1,1.50620,26.56,0.0,0.827056,False,0.827056
22,COc1ccc2cc3[n+](cc2c1OC)CCc1cc2c(cc1-3)OCO2,3.09630,40.80,0.0,0.816486,False,0.816486
70,CCCc1cn(-c2ccc(F)cc2)c2ccc(Cl)cc12,5.37550,4.93,0.0,0.812274,False,0.812274
119,CCOc1ccccc1OCC[NH2+][C@H](C)Cc1ccc(OC)c(S(N)(=...,1.31470,104.46,0.0,0.759226,False,0.759226
67,C[C@H](O)c1cn(-c2ccc(F)cc2)c2ccc(Cl)cc12,4.47630,25.16,0.0,0.734678,False,0.734678


In [36]:
herg_nn_sim = nn_tanimoto_similarity(X_herg_test[herg_fp_cols], X_herg_train[herg_fp_cols])

corr, pval = spearmanr(herg_nn_sim, herg_conf_error)
print(f"Spearman correlation (train-similarity vs. confidence error): {corr:.3f} (p={pval:.2e})")

print(f"\nMean train-similarity, misclassified:        {herg_nn_sim[~herg_correct].mean():.3f}")
print(f"Mean train-similarity, correctly classified: {herg_nn_sim[herg_correct].mean():.3f}")

Spearman correlation (train-similarity vs. confidence error): -0.074 (p=4.14e-01)

Mean train-similarity, misclassified:        0.509
Mean train-similarity, correctly classified: 0.472


### CYP3A4 — XGBoost vs. Random Forest (RF+TPSA baseline)

In [37]:
scale_pos_weight_cyp3a4 = (y_cyp3a4_train == 0).sum() / (y_cyp3a4_train == 1).sum()

xgb_cyp3a4 = XGBClassifier(
    n_estimators=200, scale_pos_weight=scale_pos_weight_cyp3a4, random_state=42, n_jobs=-1
)
xgb_cyp3a4.fit(X_cyp3a4_train, y_cyp3a4_train)

xgb_cyp3a4_metrics = evaluate_classification(xgb_cyp3a4, X_cyp3a4_test, y_cyp3a4_test)
xgb_cyp3a4_metrics['auprc'] = average_precision_score(y_cyp3a4_test, xgb_cyp3a4.predict_proba(X_cyp3a4_test)[:, 1])
print(f"AUPRC:             {xgb_cyp3a4_metrics['auprc']:.3f}")
compare_metrics('CYP3A4: Week 2 RF (no TPSA) -> XGBoost (+TPSA)', WEEK2_BASELINE['CYP3A4'], xgb_cyp3a4_metrics)
compare_metrics('CYP3A4: Week 3 RF+TPSA -> XGBoost (+TPSA)', WEEK3_RF_TPSA['CYP3A4'], xgb_cyp3a4_metrics)

ROC-AUC:           0.891
F1:                0.780
Balanced accuracy: 0.800
AUPRC:             0.874

--- CYP3A4: Week 2 RF (no TPSA) -> XGBoost (+TPSA) ---
           roc_auc: 0.879 -> 0.891  (+0.012)
                f1: 0.750 -> 0.780  (+0.030)
 balanced_accuracy: 0.779 -> 0.800  (+0.021)
             auprc: 0.852 -> 0.874  (+0.022)

--- CYP3A4: Week 3 RF+TPSA -> XGBoost (+TPSA) ---
           roc_auc: 0.878 -> 0.891  (+0.013)
                f1: 0.749 -> 0.780  (+0.031)
 balanced_accuracy: 0.779 -> 0.800  (+0.021)
             auprc: 0.851 -> 0.874  (+0.023)


### CYP3A4 — hyperparameter tuning (RF + XGBoost, scored on AUPRC)

In [38]:
# Callable scorer (not the 'average_precision' string) so both RF and
# XGBoost searches go through the same code path, rather than assuming
# sklearn's string-scorer auto-detection is bug-free for XGBoost here too
# (it wasn't for 'roc_auc' -- see the note earlier in this notebook).
def proba_auprc(estimator, X, y):
    proba = estimator.predict_proba(X)[:, 1]
    return average_precision_score(y, proba)


cyp3a4_groups = compute_scaffold_groups(cyp3a4_train_df)

rf_cyp3a4_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42),
    param_distributions=rf_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring=proba_auprc,
    random_state=42,
    n_jobs=-1,
)
rf_cyp3a4_search.fit(X_cyp3a4_train, y_cyp3a4_train, groups=cyp3a4_groups)

print(f"Best CV AUPRC: {rf_cyp3a4_search.best_score_:.3f}")
print(f"Best params: {rf_cyp3a4_search.best_params_}")

rf_cyp3a4_tuned_metrics = evaluate_classification(rf_cyp3a4_search.best_estimator_, X_cyp3a4_test, y_cyp3a4_test)
rf_cyp3a4_tuned_metrics['auprc'] = average_precision_score(
    y_cyp3a4_test, rf_cyp3a4_search.best_estimator_.predict_proba(X_cyp3a4_test)[:, 1]
)
print(f"AUPRC:             {rf_cyp3a4_tuned_metrics['auprc']:.3f}")
compare_metrics('CYP3A4: Week 2 RF -> tuned RF', WEEK2_BASELINE['CYP3A4'], rf_cyp3a4_tuned_metrics)

/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


Best CV AUPRC: 0.839
Best params: {'n_estimators': 200, 'min_samples_leaf': 1, 'max_features': 0.3, 'max_depth': 30}
ROC-AUC:           0.880
F1:                0.768
Balanced accuracy: 0.790
AUPRC:             0.856

--- CYP3A4: Week 2 RF -> tuned RF ---
           roc_auc: 0.879 -> 0.880  (+0.001)
                f1: 0.750 -> 0.768  (+0.018)
 balanced_accuracy: 0.779 -> 0.790  (+0.011)
             auprc: 0.852 -> 0.856  (+0.004)


In [39]:
xgb_cyp3a4_search = RandomizedSearchCV(
    estimator=XGBClassifier(scale_pos_weight=scale_pos_weight_cyp3a4, random_state=42),
    param_distributions=xgb_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring=proba_auprc,
    random_state=42,
    n_jobs=-1,
)
xgb_cyp3a4_search.fit(X_cyp3a4_train, y_cyp3a4_train, groups=cyp3a4_groups)

print(f"Best CV AUPRC: {xgb_cyp3a4_search.best_score_:.3f}")
print(f"Best params: {xgb_cyp3a4_search.best_params_}")

xgb_cyp3a4_tuned_metrics = evaluate_classification(xgb_cyp3a4_search.best_estimator_, X_cyp3a4_test, y_cyp3a4_test)
xgb_cyp3a4_tuned_metrics['auprc'] = average_precision_score(
    y_cyp3a4_test, xgb_cyp3a4_search.best_estimator_.predict_proba(X_cyp3a4_test)[:, 1]
)
print(f"AUPRC:             {xgb_cyp3a4_tuned_metrics['auprc']:.3f}")
compare_metrics('CYP3A4: Week 2 RF -> tuned XGBoost', WEEK2_BASELINE['CYP3A4'], xgb_cyp3a4_tuned_metrics)
compare_metrics('CYP3A4: tuned RF -> tuned XGBoost', rf_cyp3a4_tuned_metrics, xgb_cyp3a4_tuned_metrics)

/Applications/miniconda3/envs/admet/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


Best CV AUPRC: 0.860
Best params: {'subsample': 0.85, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.5}
ROC-AUC:           0.894
F1:                0.788
Balanced accuracy: 0.806
AUPRC:             0.875

--- CYP3A4: Week 2 RF -> tuned XGBoost ---
           roc_auc: 0.879 -> 0.894  (+0.015)
                f1: 0.750 -> 0.788  (+0.038)
 balanced_accuracy: 0.779 -> 0.806  (+0.027)
             auprc: 0.852 -> 0.875  (+0.023)

--- CYP3A4: tuned RF -> tuned XGBoost ---
           roc_auc: 0.880 -> 0.894  (+0.014)
                f1: 0.768 -> 0.788  (+0.020)
 balanced_accuracy: 0.790 -> 0.806  (+0.015)
             auprc: 0.856 -> 0.875  (+0.019)


### CYP3A4 — permutation importance (tuned XGBoost)

In [40]:
cyp3a4_fp_cols = [c for c in X_cyp3a4_test.columns if c.startswith('fp_')]
cyp3a4_groups_for_importance = {'Fingerprint (all 2048 bits)': cyp3a4_fp_cols}
cyp3a4_groups_for_importance.update({col: [col] for col in descriptor_cols})

def auprc_scorer(model, X, y):
    return average_precision_score(y, model.predict_proba(X)[:, 1])

cyp3a4_baseline, cyp3a4_importance = permutation_importance_grouped(
    xgb_cyp3a4_search.best_estimator_, X_cyp3a4_test, y_cyp3a4_test, cyp3a4_groups_for_importance, auprc_scorer, n_repeats=10
)
print_importance_ranking(cyp3a4_baseline, cyp3a4_importance, 'AUPRC')

Baseline AUPRC: 0.8749

 Fingerprint (all 2048 bits): importance +0.3010 (± 0.0107)
                     MolLogP: importance +0.0643 (± 0.0058)
                       MolWt: importance +0.0225 (± 0.0029)
           NumRotatableBonds: importance +0.0096 (± 0.0015)
                        TPSA: importance +0.0020 (± 0.0010)
                  NumHDonors: importance +0.0015 (± 0.0005)
               NumHAcceptors: importance +0.0011 (± 0.0007)


### CYP3A4 — failure mode analysis

In [41]:
cyp3a4_model = xgb_cyp3a4_search.best_estimator_
cyp3a4_proba = cyp3a4_model.predict_proba(X_cyp3a4_test)[:, 1]
cyp3a4_pred = (cyp3a4_proba >= 0.5).astype(int)
cyp3a4_correct = (cyp3a4_pred == y_cyp3a4_test)
cyp3a4_conf_error = np.where(y_cyp3a4_test == 1, 1 - cyp3a4_proba, cyp3a4_proba)

cyp3a4_fail_df = pd.DataFrame({
    'SMILES': cyp3a4_test_df['Drug'].values,
    'MolLogP': X_cyp3a4_test['MolLogP'].values,
    'TPSA': X_cyp3a4_test['TPSA'].values,
    'y_true': y_cyp3a4_test,
    'predicted_proba': cyp3a4_proba,
    'correct': cyp3a4_correct,
    'confidence_error': cyp3a4_conf_error,
})
cyp3a4_fail_df.sort_values('confidence_error', ascending=False).head(15)

,SMILES,MolLogP,TPSA,y_true,predicted_proba,correct,confidence_error
57,C=C(C)[C@@H]1[C@@H]2C(=O)O[C@H]1[C@H]1OC(=O)[C...,-0.86720,105.59,1,0.011846,False,0.988154
58,C=C(C)[C@H]1[C@@H]2C(=O)O[C@H]1[C@H]1OC(=O)[C@...,-0.86720,105.59,1,0.011846,False,0.988154
1237,NC(=S)c1ccc[n+]([C@@H]2O[C@@H](COP(=O)([O-])OP...,-3.62760,353.38,1,0.012894,False,0.987106
2431,O=C(O)CSCNC(=O)CC12CC3CC(CC(C3)C1)C2,2.48440,66.40,1,0.024794,False,0.975206
1911,N#Cc1c(-c2ccccc2)no[n+]1[O-],0.84668,76.76,1,0.030074,False,0.969926
215,N#Cc1c(-c2ccc3c(c2)OCO3)ccn2nc(-c3cccc(C(F)(F)...,4.68248,72.44,0,0.957130,False,0.957130
119,C=CC[C@@H]1C=C[C@H](O/N=C2\c3cc(OC)ccc3O[C@H](...,3.17970,98.97,0,0.953086,False,0.953086
1376,CCCCCCCCCC[Si](C)(C)CCC(=O)N[C@@H](Cc1ccc(C)cc...,8.63422,29.10,0,0.951734,False,0.951734
1115,CCn1c(Cc2ccccc2)nnc1SCC(=O)c1cccc([N+](=O)[O-])c1,3.77200,90.92,0,0.945731,False,0.945731
1727,COc1ccc(O[C@H]2C=C[C@@H](c3ccccc3)O[C@H]2COC(=...,6.21850,66.02,0,0.943584,False,0.943584


In [42]:
cyp3a4_nn_sim = nn_tanimoto_similarity(X_cyp3a4_test[cyp3a4_fp_cols], X_cyp3a4_train[cyp3a4_fp_cols])

corr, pval = spearmanr(cyp3a4_nn_sim, cyp3a4_conf_error)
print(f"Spearman correlation (train-similarity vs. confidence error): {corr:.3f} (p={pval:.2e})")

print(f"\nMean train-similarity, misclassified:        {cyp3a4_nn_sim[~cyp3a4_correct].mean():.3f}")
print(f"Mean train-similarity, correctly classified: {cyp3a4_nn_sim[cyp3a4_correct].mean():.3f}")

Spearman correlation (train-similarity vs. confidence error): -0.173 (p=4.24e-18)

Mean train-similarity, misclassified:        0.498
Mean train-similarity, correctly classified: 0.534


### Findings — hERG and CYP3A4

**Tuning rescues XGBoost on both properties, same as Solubility/BBB.** hERG's untuned XGBoost was actually worse than the RF baseline (ROC-AUC 0.800 vs. 0.851) — the first time XGBoost lost outright rather than being a wash — but tuned XGBoost (0.856) ends up hERG's best result, edging past both tuned RF (0.848) and the original Week 2 baseline. CYP3A4's tuned XGBoost (AUPRC 0.875) is a clean, consistent win over every prior CYP3A4 result.

**Permutation importance recovers the same known pharmacology found in Week 1 EDA, for both properties:**
- **hERG:** the fingerprint dominates, but TPSA and MolWt are close behind MolLogP as individual descriptors — consistent with Week 1's finding that MolLogP was hERG's strongest visual separator (lipophilic, basic-amine compounds are classically more prone to hERG liability).
- **CYP3A4:** MolLogP is clearly the top individual descriptor (+0.064), well ahead of every other Lipinski-style feature — matches Week 1 EDA (MolLogP was CYP3A4's clearest, if noisier, separator) and real pharmacology: CYP3A4's active site is large and hydrophobic, favoring bulky, lipophilic, aromatic-ring-rich ligands.

**The chemical-space-novelty hypothesis (Experiment 5's method, now tested on all 4 classification/regression pairs so far) still doesn't hold — and for CYP3A4 it's now a statistically strong result in the *opposite* direction.** hERG: correlation -0.074, not significant. CYP3A4: correlation -0.173, p=4.24e-18 — highly significant, but negative: misclassified molecules are *more* similar to the training set on average (0.498) than correctly-classified ones (0.534), the reverse of what "underrepresented chemical space" would predict. This is now a consistent pattern across every property tested (Solubility, BBB, hERG, CYP3A4) — worth treating as a real, replicated finding rather than one property's fluke: this scaffold-split panel's failures are not explained by test molecules being unusually dissimilar from training.

**Looking at the actual worst predictions instead shows both properties leaning on a real-but-blunt pharmacological heuristic:**
- **hERG's false positives** (predicted high blockade probability, actually non-blockers) are dominated by molecules carrying a protonated basic amine (`[NH+]`, `[NH2+]`, aromatic `[n+]`) combined with moderate-to-high MolLogP — exactly the textbook "basic amine + lipophilicity" hERG pharmacophore. The model has correctly learned this real red flag, but applies it too bluntly: many molecules that superficially match the pattern aren't actually blockers, presumably for more specific steric/electronic reasons a fingerprint+descriptor model can't resolve.
- **CYP3A4's errors run in both directions around the same "lipophilic aromatic = inhibitor" heuristic.** False positives (predicted inhibitor, isn't) are consistently high-MolLogP, aromatic/halogen-rich structures (MolLogP up to 8.6) that look like classic inhibitors but aren't. False negatives (predicted non-inhibitor, is one) include two stereoisomers of a polar natural-product-like lactone and an extremely polar nucleotide/cofactor-like molecule (TPSA=353, far outside typical drug-like range) — structurally atypical true inhibitors that don't fit the "lipophilic" pattern the model relies on. Also again spotted stereoisomer-blindness (rows 57/58 are diastereomers, same 2D descriptors, identical wrong prediction).

**Same broader lesson as Solubility/BBB, restated for these two properties:** the model has learned a real, chemically correct rule of thumb (lipophilicity/basic-amine content predicting CYP3A4/hERG liability) — Experiment 4-style permutation importance confirms it's chemically sensible — but a rule of thumb is not a universal law, and both properties' worst mistakes cluster exactly where the heuristic is necessary but not sufficient.

## Clearance: filtering censored rows before Week 3 treatment

Decided with the user before doing any further Clearance modeling: Week 1 EDA flagged that 16% of rows sit at exactly `Y=3.0` and 11% at exactly `Y=150.0` — near-certainly `<3`/`>150` assay detection-limit values, not real point measurements (a plain regressor has no way to represent "below detection limit" and just treats these as exact numbers). Rather than tune a model on top of that unresolved issue, or build a more complex censored-regression setup, the chosen fix is the simplest defensible one: **drop rows pinned at either boundary before training or evaluating**, then run Clearance through the same Experiments 2-5 treatment already given to every other property. This establishes a new baseline distinct from Week 2's and Experiment 1's Clearance numbers (both of which included the pinned rows) — comparisons below are against Week 2's original baseline and this notebook's own unfiltered RF+TPSA result, for context on how much of any change is filtering vs. TPSA vs. modeling.

In [43]:
# This notebook's own unfiltered Clearance RF+TPSA result (from Experiment 1,
# computed earlier), added as a second reference point alongside WEEK2_BASELINE.
WEEK3_RF_TPSA['Clearance'] = {'rmse': 44.119, 'mae': 34.950, 'r2': 0.129, 'spearman': 0.391}

clearance_train_filt = clearance_train_df[
    ~clearance_train_df['Y'].isin([3.0, 150.0])
].reset_index(drop=True)
clearance_test_filt = clearance_test_df[
    ~clearance_test_df['Y'].isin([3.0, 150.0])
].reset_index(drop=True)

print(f"Train: {len(clearance_train_df)} -> {len(clearance_train_filt)} rows "
      f"({len(clearance_train_df) - len(clearance_train_filt)} censored rows dropped)")
print(f"Test:  {len(clearance_test_df)} -> {len(clearance_test_filt)} rows "
      f"({len(clearance_test_df) - len(clearance_test_filt)} censored rows dropped)")

X_clearance_train_filt = featurize(clearance_train_filt)
X_clearance_test_filt = featurize(clearance_test_filt)
y_clearance_train_filt = clearance_train_filt['Y'].values
y_clearance_test_filt = clearance_test_filt['Y'].values

rf_clearance_filt = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_clearance_filt.fit(X_clearance_train_filt, y_clearance_train_filt)

rf_clearance_filt_metrics = evaluate_regression(rf_clearance_filt, X_clearance_test_filt, y_clearance_test_filt)
rf_clearance_filt_metrics['spearman'], _ = spearmanr(y_clearance_test_filt, rf_clearance_filt.predict(X_clearance_test_filt))
print(f"Spearman:          {rf_clearance_filt_metrics['spearman']:.3f}")

compare_metrics('Clearance: Week 2 baseline (unfiltered) -> RF+TPSA (filtered)', WEEK2_BASELINE['Clearance'], rf_clearance_filt_metrics)
compare_metrics('Clearance: Week 3 RF+TPSA (unfiltered) -> RF+TPSA (filtered)', WEEK3_RF_TPSA['Clearance'], rf_clearance_filt_metrics)

Train: 713 -> 547 rows (166 censored rows dropped)
Test:  203 -> 149 rows (54 censored rows dropped)


RMSE: 37.092
MAE:  27.996
R^2:  -0.030
Spearman:          0.278

--- Clearance: Week 2 baseline (unfiltered) -> RF+TPSA (filtered) ---
              rmse: 44.480 -> 37.092  (-7.388)
               mae: 35.069 -> 27.996  (-7.073)
                r2: 0.115 -> -0.030  (-0.145)
          spearman: 0.362 -> 0.278  (-0.084)

--- Clearance: Week 3 RF+TPSA (unfiltered) -> RF+TPSA (filtered) ---
              rmse: 44.119 -> 37.092  (-7.027)
               mae: 34.950 -> 27.996  (-6.954)
                r2: 0.129 -> -0.030  (-0.159)
          spearman: 0.391 -> 0.278  (-0.113)


### Clearance — XGBoost vs. Random Forest (filtered)

In [44]:
xgb_clearance_filt = XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1)
xgb_clearance_filt.fit(X_clearance_train_filt, y_clearance_train_filt)

xgb_clearance_filt_metrics = evaluate_regression(xgb_clearance_filt, X_clearance_test_filt, y_clearance_test_filt)
xgb_clearance_filt_metrics['spearman'], _ = spearmanr(y_clearance_test_filt, xgb_clearance_filt.predict(X_clearance_test_filt))
print(f"Spearman:          {xgb_clearance_filt_metrics['spearman']:.3f}")

compare_metrics('Clearance (filtered): RF -> XGBoost', rf_clearance_filt_metrics, xgb_clearance_filt_metrics)

RMSE: 38.307
MAE:  29.396
R^2:  -0.099
Spearman:          0.370

--- Clearance (filtered): RF -> XGBoost ---
              rmse: 37.092 -> 38.307  (+1.215)
               mae: 27.996 -> 29.396  (+1.401)
                r2: -0.030 -> -0.099  (-0.069)
          spearman: 0.278 -> 0.370  (+0.092)


### Clearance — hyperparameter tuning (RF + XGBoost, filtered, scored on Spearman)

In [45]:
# Clearance's leaderboard metric is Spearman correlation, which has no
# built-in sklearn scorer string -- a callable scorer computing it directly
# lets RandomizedSearchCV optimize the metric that actually matters here,
# same reasoning as MAE for Solubility and ROC-AUC for BBB/hERG.
def spearman_scorer(estimator, X, y):
    preds = estimator.predict(X)
    corr, _ = spearmanr(y, preds)
    return corr


clearance_groups = compute_scaffold_groups(clearance_train_filt)

rf_clearance_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=rf_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring=spearman_scorer,
    random_state=42,
    n_jobs=-1,
)
rf_clearance_search.fit(X_clearance_train_filt, y_clearance_train_filt, groups=clearance_groups)

print(f"Best CV Spearman: {rf_clearance_search.best_score_:.3f}")
print(f"Best params: {rf_clearance_search.best_params_}")

rf_clearance_tuned_metrics = evaluate_regression(rf_clearance_search.best_estimator_, X_clearance_test_filt, y_clearance_test_filt)
rf_clearance_tuned_metrics['spearman'], _ = spearmanr(y_clearance_test_filt, rf_clearance_search.best_estimator_.predict(X_clearance_test_filt))
print(f"Spearman:          {rf_clearance_tuned_metrics['spearman']:.3f}")
compare_metrics('Clearance (filtered): RF -> tuned RF', rf_clearance_filt_metrics, rf_clearance_tuned_metrics)

Best CV Spearman: 0.364
Best params: {'n_estimators': 200, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 30}
RMSE: 34.213
MAE:  27.037
R^2:  0.123
Spearman:          0.346

--- Clearance (filtered): RF -> tuned RF ---
              rmse: 37.092 -> 34.213  (-2.879)
               mae: 27.996 -> 27.037  (-0.959)
                r2: -0.030 -> 0.123  (+0.154)
          spearman: 0.278 -> 0.346  (+0.068)


In [46]:
xgb_clearance_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_distributions=xgb_param_grid,
    n_iter=15,
    cv=GroupKFold(n_splits=5),
    scoring=spearman_scorer,
    random_state=42,
    n_jobs=-1,
)
xgb_clearance_search.fit(X_clearance_train_filt, y_clearance_train_filt, groups=clearance_groups)

print(f"Best CV Spearman: {xgb_clearance_search.best_score_:.3f}")
print(f"Best params: {xgb_clearance_search.best_params_}")

xgb_clearance_tuned_metrics = evaluate_regression(xgb_clearance_search.best_estimator_, X_clearance_test_filt, y_clearance_test_filt)
xgb_clearance_tuned_metrics['spearman'], _ = spearmanr(y_clearance_test_filt, xgb_clearance_search.best_estimator_.predict(X_clearance_test_filt))
print(f"Spearman:          {xgb_clearance_tuned_metrics['spearman']:.3f}")
compare_metrics('Clearance (filtered): tuned RF -> tuned XGBoost', rf_clearance_tuned_metrics, xgb_clearance_tuned_metrics)
compare_metrics('Clearance (filtered): RF baseline -> tuned XGBoost', rf_clearance_filt_metrics, xgb_clearance_tuned_metrics)

Best CV Spearman: 0.359
Best params: {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 9, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
RMSE: 34.404
MAE:  26.699
R^2:  0.114
Spearman:          0.381

--- Clearance (filtered): tuned RF -> tuned XGBoost ---
              rmse: 34.213 -> 34.404  (+0.191)
               mae: 27.037 -> 26.699  (-0.338)
                r2: 0.123 -> 0.114  (-0.010)
          spearman: 0.346 -> 0.381  (+0.035)

--- Clearance (filtered): RF baseline -> tuned XGBoost ---
              rmse: 37.092 -> 34.404  (-2.688)
               mae: 27.996 -> 26.699  (-1.297)
                r2: -0.030 -> 0.114  (+0.144)
          spearman: 0.278 -> 0.381  (+0.103)


### Clearance — permutation importance (tuned XGBoost, filtered, scored on Spearman)

In [47]:
clearance_fp_cols = [c for c in X_clearance_test_filt.columns if c.startswith('fp_')]
clearance_groups_for_importance = {'Fingerprint (all 2048 bits)': clearance_fp_cols}
clearance_groups_for_importance.update({col: [col] for col in descriptor_cols})

clearance_baseline, clearance_importance = permutation_importance_grouped(
    xgb_clearance_search.best_estimator_, X_clearance_test_filt, y_clearance_test_filt,
    clearance_groups_for_importance, spearman_scorer, n_repeats=10
)
print_importance_ranking(clearance_baseline, clearance_importance, 'Spearman')

Baseline Spearman: 0.3811

 Fingerprint (all 2048 bits): importance +0.3320 (± 0.0909)
                       MolWt: importance +0.0404 (± 0.0204)
                        TPSA: importance +0.0125 (± 0.0209)
               NumHAcceptors: importance +0.0019 (± 0.0036)
                     MolLogP: importance +0.0005 (± 0.0097)
                  NumHDonors: importance -0.0037 (± 0.0027)
           NumRotatableBonds: importance -0.0063 (± 0.0033)


### Clearance — failure mode analysis (filtered)

In [48]:
clearance_model = xgb_clearance_search.best_estimator_
clearance_pred = clearance_model.predict(X_clearance_test_filt)
clearance_abs_err = np.abs(clearance_pred - y_clearance_test_filt)

clearance_fail_df = pd.DataFrame({
    'SMILES': clearance_test_filt['Drug'].values,
    'MolWt': X_clearance_test_filt['MolWt'].values,
    'MolLogP': X_clearance_test_filt['MolLogP'].values,
    'TPSA': X_clearance_test_filt['TPSA'].values,
    'y_true': y_clearance_test_filt,
    'y_pred': clearance_pred,
    'abs_error': clearance_abs_err,
})
clearance_fail_df.sort_values('abs_error', ascending=False).head(15)

,SMILES,MolWt,MolLogP,TPSA,y_true,y_pred,abs_error
18,CC(C)[C@H]1C(=O)N[C@H](CO)Cc2ccccc2N1C,262.353,1.18060,52.57,144.540,30.910368,113.629632
44,COCCOc1nc(N)c2[nH]c(=O)n(Cc3cccc(CC(=O)OC)c3)c2n1,387.396,0.49080,134.35,125.000,25.580244,99.419756
118,Nc1ccc2c(-c3ccncc3)c(-c3ccc(F)cc3)[nH]c2n1,304.328,4.01320,67.59,117.490,24.181471,93.308529
50,COc1cc(C(=O)OCCCCNC(=N)N)cc(OC)c1O,311.338,0.82947,126.89,138.040,46.121418,91.918582
33,CCOC(=O)c1c(-c2ccccc2)nc2cc(C)ccn12,280.327,3.48642,43.60,147.910,62.871964,85.038036
88,Cc1cc(C)nc(SCC(=O)Nc2cc(C)on2)n1,278.337,2.12066,80.91,141.250,59.795868,81.454132
70,C[C@@H](NCc1ccccc1-c1ccc(CCNC[C@H](O)c2ccc(O)c...,539.701,5.67870,97.38,108.000,29.880686,78.119314
148,c1ccc(Nc2[nH]nnc2-c2ccccc2)cc1,236.278,3.21530,53.60,111.000,40.199326,70.800674
96,Cc1ccc(C)c(OCCCC(C)(C)C(=O)O)c1,250.338,3.57324,46.53,108.035,37.537762,70.497238
20,CC1Cc2ccccc2N1NC(=O)c1ccc(Cl)c(S(N)(=O)=O)c1,365.842,2.08340,92.50,9.040,75.101288,66.061288


In [49]:
clearance_nn_sim = nn_tanimoto_similarity(X_clearance_test_filt[clearance_fp_cols], X_clearance_train_filt[clearance_fp_cols])

corr, pval = spearmanr(clearance_nn_sim, clearance_abs_err)
print(f"Spearman correlation (train-similarity vs. abs error): {corr:.3f} (p={pval:.2e})")

worst_10pct = clearance_fail_df['abs_error'] >= clearance_fail_df['abs_error'].quantile(0.90)
print(f"\nMean train-similarity, worst 10% of predictions: {clearance_nn_sim[worst_10pct.values].mean():.3f}")
print(f"Mean train-similarity, remaining 90%:             {clearance_nn_sim[~worst_10pct.values].mean():.3f}")

Spearman correlation (train-similarity vs. abs error): -0.153 (p=6.30e-02)

Mean train-similarity, worst 10% of predictions: 0.402
Mean train-similarity, remaining 90%:             0.416


### Findings — filtering made the raw number *worse*, and that's worth understanding, not ignoring

**The headline number went down, not up.** Untuned filtered RF: Spearman 0.278, vs. 0.391 for the unfiltered RF+TPSA baseline (Experiment 1) — and R² actually went negative (-0.030), worse than just predicting the mean. Even after a full tuning + XGBoost pass, the best filtered result (Spearman 0.381) still falls just short of the original unfiltered number. Reporting this honestly rather than quietly picking whichever number looks better.

**Digging into why, rather than stopping at "filtering didn't work":**

1. **The comparison isn't quite apples-to-apples — the test set itself changed.** Filtering didn't just change training data, it also removed 54 of 203 test molecules (the ones pinned at Y=3/150). Those pinned test points are comparatively easy for a rank-correlation metric: many molecules really do sit at the assay floor or ceiling, and a model predicting "low" or "high" for them scores reasonably well on Spearman even without learning much. The unfiltered Spearman of 0.391 was partly inflated by these "gimme" points. The filtered Spearman of 0.381 measures something harder and more honest: rank agreement purely within the continuous, non-boundary range — a fairer test of whether the model has learned real structure-clearance relationships, not just "can it spot the obvious extremes."
2. **Permutation importance explains where TPSA's earlier importance actually came from.** Experiment 1 found TPSA was Clearance's #1 most important feature by MDI (importance 0.073, ahead of every other feature in a 2054-column model) — the strongest TPSA signal of any property in the whole panel. On the *filtered* data, TPSA drops to a distant third individual descriptor (+0.013), far behind the fingerprint (+0.332) and MolWt (+0.040). The likely explanation: TPSA is very good at flagging *which* molecules are censored (very polar, high-TPSA compounds plausibly cluster at the low-clearance floor due to poor hepatocyte membrane permeability; lipophilic low-TPSA compounds plausibly cluster at the high-clearance ceiling) — so most of TPSA's predictive power in the unfiltered model was about identifying the censored extremes, not about explaining clearance variation within the normal range. Filtering removed exactly the part of the problem TPSA was good at, leaving a harder residual problem where the fingerprint has to do most of the work.
3. **The worst-predicted molecules (even after filtering) are still concentrated at genuinely high true clearance values** (several >100, up to 147.9) that the model consistently under-predicts — the difficulty of extrapolating to the high end wasn't really about the *exact*-pinned Y=150 rows; it's a broader small-sample-at-the-extremes problem that filtering doesn't fix, since it only removes the exactly-pinned points, not the general scarcity of high-clearance training examples.
4. **The chemical-space-novelty hypothesis is, once again, not supported** (correlation -0.153, p=0.063, borderline but not significant, same "wrong direction" pattern as every other property tested) — the 5th property in a row where this doesn't hold.

**Bottom line:** filtering was still the right call, even though the number went down. The unfiltered Spearman of 0.391 was measuring a mix of "can the model rank real clearance variation" and "can the model spot assay-boundary artifacts" — the filtered Spearman of 0.381 measures only the first, harder thing. A lower number on a more honestly-defined problem is a more meaningful result than a higher number partly inflated by an artifact this project explicitly identified and decided to correct.

## Week 5: saving the final tuned models for the Streamlit demo app

Saves each property's Experiment 3 tuned-XGBoost `best_estimator_` (the same 5 final numbers reported in the README's results table — Clearance uses the filtered dataset, per the decision above) to disk via `joblib`, so `app.py` (Week 5's Streamlit demo) can load them without retraining.

In [50]:
import os
import joblib

os.makedirs('models', exist_ok=True)

joblib.dump(xgb_sol_search.best_estimator_, 'models/solubility_xgb.pkl')
joblib.dump(xgb_bbb_search.best_estimator_, 'models/bbb_xgb.pkl')
joblib.dump(xgb_herg_search.best_estimator_, 'models/herg_xgb.pkl')
joblib.dump(xgb_cyp3a4_search.best_estimator_, 'models/cyp3a4_xgb.pkl')
joblib.dump(xgb_clearance_search.best_estimator_, 'models/clearance_xgb.pkl')

print("Saved 5 tuned XGBoost models to models/")
for f in sorted(os.listdir('models')):
    print(f"  {f}")


Saved 5 tuned XGBoost models to models/
  bbb_xgb.pkl
  clearance_xgb.pkl
  cyp3a4_xgb.pkl
  herg_xgb.pkl
  solubility_xgb.pkl
